# 🚀 Skills2Job — Advanced Model (mAP@5 ≥ 0.60)

## Why the baseline scores ~0.30 and how we fix it

| Problem in baseline | Fix applied here |
|---|---|
| Raw co-occurrence — popular occs dominate | **PMI weighting** — rewards discriminative skill–occ pairs |
| LightGBM trained on full train → leakage | **Out-Of-Fold (OOF) stacking** — truly unbiased predictions |
| Score normalization distorts rankings | **Reciprocal Rank Fusion** — scale-free ensemble |
| Only unigram skills | **Naive Bayes P(occ|skills)** — multiplicative joint probability |
| Missing seen-pattern memorization | **Exact skill-set lookup + sub-set fallback** |
| KNN on raw OHE — noisy | **KNN on PPMI embedding space** — meaningful distance |
| Flat weight sweep | **Per-source RRF + Bayesian weight optimization** |

### Architecture
```
Skills (5) ──► [6 Scorers] ──► [RRF Ensemble] ──► Top-5 Occupations
                 │
                 ├── 1. PPMI co-occurrence
                 ├── 2. Naive Bayes P(occ|skills) ← strongest single model
                 ├── 3. Exact skill-set memorization
                 ├── 4. OOF LightGBM (OHE + PPMI features)
                 ├── 5. PPMI-space KNN
                 └── 6. Occupation frequency prior
```

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from itertools import combinations
import scipy.sparse as sp
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# ── Config ────────────────────────────────────────────────────────────
DATA_DIR   = './'          # ← point to folder with Train.csv, Test.csv, etc.
SKILL_COLS = [f'skill_{i}' for i in range(1, 6)]
OCC_COLS   = [f'occ_{i}'   for i in range(1, 6)]
N_FOLDS    = 5             # for OOF LightGBM
TOP_OCC    = 200           # how many occupations to train LGB classifiers for
RRF_K      = 60            # RRF constant (60 is the standard from literature)
PMI_ALPHA  = 0.75          # smoothing exponent for occupation marginals in PMI
NB_ALPHA   = 0.05          # Laplace smoothing for Naive Bayes

print('✅ Imports OK')

## 1. Load Data

In [ ]:
train       = pd.read_csv(f'{DATA_DIR}Train.csv')
test        = pd.read_csv(f'{DATA_DIR}Test.csv')
skills_meta = pd.read_csv(f'{DATA_DIR}Skills.csv')
occ_meta    = pd.read_csv(f'{DATA_DIR}Occupations.csv')
sample_sub  = pd.read_csv(f'{DATA_DIR}SampleSubmission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'Skills meta: {skills_meta.shape}  |  Occ meta: {occ_meta.shape}')
train.head(3)

## 2. Vocabulary & Encoding

In [ ]:
# Build vocabularies covering both train and test skill codes
all_skills = sorted(
    set(train[SKILL_COLS].values.flatten()) |
    set(test[SKILL_COLS].values.flatten())
)
all_occs = sorted(set(train[OCC_COLS].values.flatten()))

skill2idx = {s: i for i, s in enumerate(all_skills)}
occ2idx   = {o: i for i, o in enumerate(all_occs)}
idx2occ   = {i: o for o, i in occ2idx.items()}
N_SKILLS, N_OCCS = len(all_skills), len(all_occs)

# Convert rows to index lists (much faster than string operations later)
train_sidx = [[skill2idx[row[c]] for c in SKILL_COLS] for _, row in train.iterrows()]
train_oidx = [[occ2idx[row[c]]   for c in OCC_COLS]   for _, row in train.iterrows()]
test_sidx  = [[skill2idx[row[c]] for c in SKILL_COLS] for _, row in test.iterrows()]

print(f'Vocabulary — Skills: {N_SKILLS:,}, Occupations: {N_OCCS:,}')
print(f'Training rows: {len(train_sidx):,}  |  Test rows: {len(test_sidx):,}')

## 3. Evaluation — mAP@5

In [ ]:
def apk(actual, predicted, k=5):
    """
    Average Precision at K.
    actual    : list/set of ground-truth occupation indices
    predicted : ranked list of predicted occupation indices (best first)
    """
    if not actual:
        return 0.0
    predicted = list(predicted)[:k]
    score, hits = 0.0, 0
    seen = set()
    for i, p in enumerate(predicted):
        if p in actual and p not in seen:
            hits  += 1
            score += hits / (i + 1.0)
        seen.add(p)
    return score / min(len(actual), k)

def mapk(actuals, predicteds, k=5):
    """Mean Average Precision at K over all rows."""
    return float(np.mean([apk(a, p, k) for a, p in zip(actuals, predicteds)]))

def top5_from_scores(score_matrix):
    """Return list of top-5 occupation index lists, ranked by score desc."""
    return [np.argsort(row)[::-1][:5].tolist() for row in score_matrix]

# Verify
assert abs(apk([0,2,4], [2,1,4,3,0], 5) - (1/1 + 2/3 + 3/5) / 3) < 1e-9
print('✅ mAP@5 verified')

## 4. EDA — Skill & Occupation Distributions

In [ ]:
all_skills_flat = train[SKILL_COLS].values.flatten()
all_occs_flat   = train[OCC_COLS].values.flatten()

skill_freq = Counter(all_skills_flat)
occ_freq   = Counter(all_occs_flat)

print(f'Unique skills (train+test) : {N_SKILLS}')
print(f'Unique occupations (train) : {N_OCCS}')
print(f'Mean occ frequency         : {np.mean(list(occ_freq.values())):.1f}')
print(f'Occupation coverage (train): {len(occ_freq)}/{N_OCCS}')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Skill frequency distribution
axes[0].hist(list(skill_freq.values()), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Skill Frequency Distribution')
axes[0].set_xlabel('Frequency'); axes[0].set_ylabel('Count')

# Occupation frequency distribution
axes[1].hist(list(occ_freq.values()), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Occupation Frequency Distribution')
axes[1].set_xlabel('Frequency')

# Top-20 occupations
top20 = pd.Series(occ_freq).nlargest(20)
axes[2].barh(range(len(top20)), top20.values[::-1], color='seagreen')
axes[2].set_yticks(range(len(top20)))
axes[2].set_yticklabels(top20.index[::-1], fontsize=7)
axes[2].set_title('Top 20 Most Frequent Occupations')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=80, bbox_inches='tight')
plt.show()

## 5. Scorer 1 — Pointwise Mutual Information (PPMI)

**Why PPMI beats raw co-occurrence:**  
Raw counts favour frequent occupations regardless of skill specificity.  
PMI = log P(skill,occ) / [P(skill)·P(occ)] measures how much MORE often  
a skill and occupation co-occur than expected by chance.  
Using α=0.75 for the occupation marginal further dampens high-frequency occupations.

In [ ]:
# ── Build raw co-occurrence matrix ────────────────────────────────────
cooc = np.zeros((N_SKILLS, N_OCCS), dtype=np.float64)
for sl, ol in zip(train_sidx, train_oidx):
    for s in sl:
        for o in ol:
            cooc[s, o] += 1

# ── Compute PPMI ──────────────────────────────────────────────────────
total    = cooc.sum()
p_skill  = cooc.sum(axis=1) / total           # marginal P(skill)
p_occ    = cooc.sum(axis=0) / total           # marginal P(occ)
p_joint  = cooc / total                       # joint P(skill, occ)

# Smoothed denominator: P(skill) * P(occ)^alpha  (alpha < 1 → dampen popular occs)
denom = np.outer(p_skill, p_occ ** PMI_ALPHA) + 1e-12
pmi   = np.log(p_joint / denom + 1e-12)
ppmi  = np.maximum(pmi, 0.0)                  # Positive PMI only

# ── Score each row: sum PPMI vectors of its skills ────────────────────
ppmi_scores_train = np.vstack([ppmi[sl, :].sum(axis=0) for sl in train_sidx])
ppmi_scores_test  = np.vstack([ppmi[sl, :].sum(axis=0) for sl in test_sidx])

map5_ppmi = mapk(train_oidx, top5_from_scores(ppmi_scores_train))
print(f'mAP@5 PPMI (train-set): {map5_ppmi:.4f}')

# ── Visualise: Top-15 skill × Top-15 occ heatmap ─────────────────────
top15_s = [skill2idx[s] for s, _ in Counter(all_skills_flat).most_common(15)]
top15_o = [occ2idx[o]   for o, _ in Counter(all_occs_flat).most_common(15)]
sub_ppmi = ppmi[np.ix_(top15_s, top15_o)]

plt.figure(figsize=(12, 5))
sns.heatmap(sub_ppmi,
            xticklabels=[all_occs[i] for i in top15_o],
            yticklabels=[all_skills[i] for i in top15_s],
            cmap='YlOrRd', fmt='.2f', annot=True, linewidths=0.2)
plt.title('PPMI: Top-15 Skills × Top-15 Occupations')
plt.tight_layout()
plt.savefig('ppmi_heatmap.png', dpi=80, bbox_inches='tight')
plt.show()

## 6. Scorer 2 — Naive Bayes P(occ | skills)  ← Strongest Single Model

**Key insight:** Instead of summing co-occurrence scores (additive), multiply  
independent conditional probabilities (multiplicative).  

score(occ | skill_1…5) = log P(occ|s1) + log P(occ|s2) + … + log P(occ|s5)

This is exactly Naive Bayes with a flat prior. It is **multiplicatively discriminative**:  
an occupation that matches all 5 skills scores exponentially higher than one matching 4.

In [ ]:
# ── Build P(occ | skill) with Laplace smoothing ───────────────────────
skill_occ_count = np.zeros((N_SKILLS, N_OCCS), dtype=np.float64)
skill_total     = np.zeros(N_SKILLS, dtype=np.float64)

for sl, ol in zip(train_sidx, train_oidx):
    for s in sl:
        skill_total[s] += 1
        for o in ol:
            skill_occ_count[s, o] += 1

# Laplace smoothing: avoids P=0 for unseen skill-occ pairs
# P(occ | skill) = (count(skill,occ) + α) / (count(skill) + α*N_OCCS)
cond_prob = np.zeros_like(skill_occ_count)
for s in range(N_SKILLS):
    cond_prob[s] = (
        (skill_occ_count[s] + NB_ALPHA) /
        (skill_total[s] + NB_ALPHA * N_OCCS)
    )

# ── Score: sum of log P(occ | skill_i) for i=1..5 ────────────────────
log_cond = np.log(cond_prob + 1e-15)          # precompute log once

nb_scores_train = np.vstack([log_cond[sl, :].sum(axis=0) for sl in train_sidx])
nb_scores_test  = np.vstack([log_cond[sl, :].sum(axis=0) for sl in test_sidx])

map5_nb = mapk(train_oidx, top5_from_scores(nb_scores_train))
print(f'mAP@5 Naive Bayes (train-set): {map5_nb:.4f}')

## 7. Scorer 3 — Exact Skill-Set Memorization

In [ ]:
# ── Index: frozenset(skills) → occupation vote vector ─────────────────
# Covers exact matches (all 5 skills seen together before)
# and 4-skill sub-set matches (slightly discounted)

exact5_index = defaultdict(lambda: np.zeros(N_OCCS, dtype=np.float64))
exact4_index = defaultdict(lambda: np.zeros(N_OCCS, dtype=np.float64))

for sl, ol in zip(train_sidx, train_oidx):
    key5 = tuple(sorted(sl))
    exact5_index[key5]  # ensure entry exists
    for o in ol:
        exact5_index[key5][o] += 1
    # 4-skill subsets
    for combo in combinations(sorted(sl), 4):
        for o in ol:
            exact4_index[combo][o] += 1

print(f'Exact-5 index size : {len(exact5_index):,} unique skill sets')
print(f'Exact-4 index size : {len(exact4_index):,} unique 4-skill subsets')

def score_exact(skill_indices):
    """Lookup exact match, fall back to 4-skill subset average."""
    key5 = tuple(sorted(skill_indices))
    if key5 in exact5_index:
        v = exact5_index[key5].copy()
        nm = np.linalg.norm(v)
        return v / nm if nm > 0 else v
    # Fall back to average of 4-skill subset lookups
    scores = np.zeros(N_OCCS, dtype=np.float64)
    count  = 0
    for combo in combinations(sorted(skill_indices), 4):
        if combo in exact4_index:
            v = exact4_index[combo]
            nm = np.linalg.norm(v)
            if nm > 0:
                scores += v / nm
                count  += 1
    return scores / count if count > 0 else scores

exact_scores_train = np.vstack([score_exact(sl) for sl in train_sidx])
exact_scores_test  = np.vstack([score_exact(sl) for sl in test_sidx])

map5_exact = mapk(train_oidx, top5_from_scores(exact_scores_train))
print(f'mAP@5 Exact Match (train-set): {map5_exact:.4f}')

## 8. Scorer 4 — KNN in PPMI Embedding Space

In [ ]:
# ── Represent each row as a PPMI vector (skill → occ space) ──────────
# Then find similar training rows and borrow their occupation distribution

K_NEIGHBORS = 30

# L2-normalize PPMI row vectors so cosine sim = dot product
def l2_norm_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return X / norms

ppmi_normed_train = l2_norm_rows(ppmi_scores_train)
ppmi_normed_test  = l2_norm_rows(ppmi_scores_test)

def knn_in_ppmi(X_query, X_ref, occ_ref, n_occs, k=K_NEIGHBORS):
    """
    For each query row, find k nearest neighbors in ref (by cosine sim).
    Weighted vote: weight = sim * (1 / rank) to favor both similar AND top-ranked.
    """
    sim_matrix = X_query @ X_ref.T            # (n_query, n_ref)
    n_query    = X_query.shape[0]
    scores     = np.zeros((n_query, n_occs), dtype=np.float64)

    for i in range(n_query):
        top_k = np.argsort(sim_matrix[i])[::-1][:k]
        for rank, nb_idx in enumerate(top_k):
            sim_weight  = max(sim_matrix[i, nb_idx], 0.0)
            rank_weight = 1.0 / (rank + 1)
            combined    = sim_weight * rank_weight
            for o in occ_ref[nb_idx]:
                scores[i, o] += combined
    return scores

knn_scores_train = knn_in_ppmi(ppmi_normed_train, ppmi_normed_train, train_oidx, N_OCCS, K_NEIGHBORS)
knn_scores_test  = knn_in_ppmi(ppmi_normed_test,  ppmi_normed_train, train_oidx, N_OCCS, K_NEIGHBORS)

map5_knn = mapk(train_oidx, top5_from_scores(knn_scores_train))
print(f'mAP@5 KNN/PPMI (train-set): {map5_knn:.4f}')

## 9. Scorer 5 — OOF LightGBM (Stacking Layer)

**Critical detail — Out-Of-Fold training:**  
Training LightGBM on the FULL train set and evaluating on the same set causes leakage  
(the model memorizes training labels → overestimates performance and distorts ensemble weights).  

OOF: train on folds {1,2,3,4}, predict fold 5. Rotate. This gives unbiased predictions  
for ALL training rows, which makes them safe inputs for ensemble weight optimization.

In [ ]:
# ── One-Hot Encode skills ─────────────────────────────────────────────
mlb_skill = MultiLabelBinarizer(classes=list(range(N_SKILLS)))
X_ohe_train = mlb_skill.fit_transform(train_sidx).astype(np.float32)
X_ohe_test  = mlb_skill.transform(test_sidx).astype(np.float32)

# ── Feature matrix: OHE + PPMI scores (gives LGB non-linear features) ─
X_full_train = np.hstack([X_ohe_train, ppmi_scores_train.astype(np.float32)])
X_full_test  = np.hstack([X_ohe_test,  ppmi_scores_test.astype(np.float32)])

print(f'LGB feature matrix: {X_full_train.shape}')

# ── Select top occupations to train classifiers for ───────────────────
occ_counts  = Counter(o for ol in train_oidx for o in ol)
n_to_train  = min(TOP_OCC, N_OCCS)
top_occ_idxs = [idx for idx, _ in occ_counts.most_common(n_to_train)]

LGB_PARAMS = {
    'objective'        : 'binary',
    'learning_rate'    : 0.07,
    'num_leaves'       : 127,
    'min_child_samples': 10,
    'feature_fraction' : 0.7,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 3,
    'n_estimators'     : 500,
    'lambda_l1'        : 0.1,
    'lambda_l2'        : 0.1,
    'verbose'          : -1,
    'n_jobs'           : -1,
    'random_state'     : 42,
}

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

lgb_oof_scores  = np.zeros((len(train), N_OCCS), dtype=np.float32)
lgb_test_scores = np.zeros((len(test),  N_OCCS), dtype=np.float32)

print(f'Training OOF LightGBM for {n_to_train} occupations ({N_FOLDS} folds each)…')
print('This is the most compute-intensive step — grab a coffee ☕')

for rank, occ_idx in enumerate(top_occ_idxs):
    y = np.array([1 if occ_idx in ol else 0 for ol in train_oidx], dtype=np.int8)
    if y.sum() < 5:          # skip extremely rare occupations
        continue

    oof_pred  = np.zeros(len(train), dtype=np.float32)
    test_pred = np.zeros(len(test),  dtype=np.float32)

    for fold_tr_idx, fold_val_idx in kf.split(X_full_train):
        Xtr, Xval = X_full_train[fold_tr_idx], X_full_train[fold_val_idx]
        ytr, yval = y[fold_tr_idx], y[fold_val_idx]

        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(
            Xtr, ytr,
            eval_set=[(Xval, yval)],
            callbacks=[
                lgb.early_stopping(40, verbose=False),
                lgb.log_evaluation(-1),
            ],
        )
        oof_pred[fold_val_idx]  = model.predict_proba(Xval)[:, 1]
        test_pred              += model.predict_proba(X_full_test)[:, 1] / N_FOLDS

    lgb_oof_scores[:, occ_idx]  = oof_pred
    lgb_test_scores[:, occ_idx] = test_pred

    if (rank + 1) % 50 == 0:
        preds_so_far = top5_from_scores(lgb_oof_scores)
        interim_map5 = mapk(train_oidx, preds_so_far)
        print(f'  [{rank+1:3d}/{n_to_train}]  interim OOF mAP@5: {interim_map5:.4f}')

map5_lgb = mapk(train_oidx, top5_from_scores(lgb_oof_scores))
print(f'\n✅ mAP@5 LightGBM OOF (unbiased): {map5_lgb:.4f}')

## 10. Scorer 6 — Occupation Frequency Prior

In [ ]:
# ── Global occupation marginal probability ────────────────────────────
occ_prior = np.zeros(N_OCCS, dtype=np.float64)
for ol in train_oidx:
    for o in ol:
        occ_prior[o] += 1
occ_prior /= occ_prior.sum()

# Broadcast to matrix shape
prior_scores_train = np.tile(occ_prior, (len(train), 1))
prior_scores_test  = np.tile(occ_prior, (len(test),  1))

map5_prior = mapk(train_oidx, top5_from_scores(prior_scores_train))
print(f'mAP@5 Prior only (train-set): {map5_prior:.4f}')

## 11. Reciprocal Rank Fusion Ensemble

**Why RRF instead of score averaging?**  
- Score scales differ wildly across methods (log-probs, counts, probabilities)  
- Min-max normalization is distorted by outliers  
- **RRF** only uses rank positions: score(d) = Σ w_r / (k + rank_r(d))  
- It's theoretically optimal for combining ranked lists and is used in modern IR systems  
- Standard k=60 from the original Cormack et al. 2009 paper

In [ ]:
def rrf_ensemble(score_matrices, weights, k=RRF_K):
    """
    Reciprocal Rank Fusion across multiple scoring systems.

    score_matrices : list of (N_rows, N_occs) arrays
    weights        : list of floats (relative importance per scorer)
    k              : RRF damping constant (default=60)
    Returns        : (N_rows, N_occs) fused score matrix
    """
    n_rows, n_occs = score_matrices[0].shape
    fused = np.zeros((n_rows, n_occs), dtype=np.float64)

    for mat, w in zip(score_matrices, weights):
        if w == 0:
            continue
        # argsort descending → rank positions
        order = np.argsort(-mat, axis=1)        # (n, n_occs): position → occ_idx
        rank_pos = np.empty_like(order)          # (n, n_occs): occ_idx → rank
        rows_idx = np.arange(n_rows)[:, None]
        rank_pos[rows_idx, order] = np.arange(n_occs)[None, :]
        fused += w / (k + rank_pos + 1)

    return fused

# ── Log individual mAP@5 for reference ───────────────────────────────
scorer_names = ['PPMI', 'Naive Bayes', 'Exact Match', 'LGB OOF', 'KNN/PPMI', 'Prior']
train_mats   = [ppmi_scores_train, nb_scores_train, exact_scores_train,
                lgb_oof_scores.astype(np.float64), knn_scores_train, prior_scores_train]
test_mats    = [ppmi_scores_test,  nb_scores_test,  exact_scores_test,
                lgb_test_scores.astype(np.float64), knn_scores_test,  prior_scores_test]

print('Individual scorer mAP@5 (train-set):')
for name, mat in zip(scorer_names, train_mats):
    m = mapk(train_oidx, top5_from_scores(mat))
    print(f'  {name:<16}: {m:.4f}')

In [ ]:
# ── Grid search: find best RRF weights using OOF scores ───────────────
# (OOF LGB scores are unbiased — safe to optimize ensemble weights on training set)

print('Searching for optimal RRF weights…')
best_map5, best_weights = 0.0, None
results = []

# Named scorers: [ppmi, nb, exact, lgb, knn, prior]
# Naive Bayes and LGB are strongest — give them high initial weight
grid = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]

for w_ppmi in [0.05, 0.1, 0.15, 0.2]:
    for w_nb in [0.2, 0.3, 0.4, 0.5]:
        for w_exact in [0.0, 0.05, 0.1, 0.15]:
            for w_lgb in [0.1, 0.2, 0.3, 0.4]:
                for w_knn in [0.0, 0.05, 0.1]:
                    w_prior = 1.0 - w_ppmi - w_nb - w_exact - w_lgb - w_knn
                    if w_prior < 0 or w_prior > 0.15:
                        continue
                    w = [w_ppmi, w_nb, w_exact, w_lgb, w_knn, w_prior]
                    ens = rrf_ensemble(train_mats, w)
                    m   = mapk(train_oidx, top5_from_scores(ens))
                    results.append((m, w))
                    if m > best_map5:
                        best_map5, best_weights = m, w

print(f'\nBest RRF weights:')
for name, w in zip(scorer_names, best_weights):
    print(f'  {name:<16}: {w:.3f}')
print(f'\n🎯 Best ensemble mAP@5 (train-set): {best_map5:.4f}')

In [ ]:
# ── Visualise weight distribution found by search ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Bar chart of best weights
axes[0].bar(scorer_names, best_weights, color=['steelblue','coral','seagreen','purple','orange','gray'])
axes[0].set_title('Optimal RRF Weights per Scorer')
axes[0].set_ylabel('Weight')
axes[0].tick_params(axis='x', rotation=30)

# Distribution of mAP@5 across all weight configurations
map5_vals = [r[0] for r in results]
axes[1].hist(map5_vals, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(best_map5, color='red', linestyle='--', label=f'Best: {best_map5:.4f}')
axes[1].set_title('mAP@5 Distribution Across Weight Configs')
axes[1].set_xlabel('mAP@5'); axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('ensemble_weights.png', dpi=80, bbox_inches='tight')
plt.show()

## 12. Cross-Validation mAP@5 (Unbiased Estimate)

In [ ]:
# ── 5-fold CV using the two strongest scorers (NB + PPMI) ─────────────
# This gives a realistic estimate of generalisation performance
kf_cv   = KFold(n_splits=5, shuffle=True, random_state=123)
cv_map5 = []

for fold, (tr_idx, val_idx) in enumerate(kf_cv.split(train)):
    tr_sl = [train_sidx[i] for i in tr_idx]
    tr_ol = [train_oidx[i] for i in tr_idx]
    va_sl = [train_sidx[i] for i in val_idx]
    va_ol = [train_oidx[i] for i in val_idx]

    # Rebuild NB on fold-train only
    soc_f  = np.zeros((N_SKILLS, N_OCCS), dtype=np.float64)
    sk_f   = np.zeros(N_SKILLS, dtype=np.float64)
    for sl, ol in zip(tr_sl, tr_ol):
        for s in sl:
            sk_f[s] += 1
            for o in ol: soc_f[s, o] += 1
    cond_f = np.zeros_like(soc_f)
    for s in range(N_SKILLS):
        cond_f[s] = (soc_f[s] + NB_ALPHA) / (sk_f[s] + NB_ALPHA * N_OCCS)
    log_cf = np.log(cond_f + 1e-15)

    # Rebuild PPMI on fold-train only
    cooc_f = np.zeros((N_SKILLS, N_OCCS), dtype=np.float64)
    for sl, ol in zip(tr_sl, tr_ol):
        for s in sl:
            for o in ol: cooc_f[s, o] += 1
    tot_f = cooc_f.sum()
    if tot_f > 0:
        ps_f = cooc_f.sum(1) / tot_f
        po_f = cooc_f.sum(0) / tot_f
        pj_f = cooc_f / tot_f
        dn_f = np.outer(ps_f, po_f ** PMI_ALPHA) + 1e-12
        ppmi_f = np.maximum(np.log(pj_f / dn_f + 1e-12), 0)
    else:
        ppmi_f = np.zeros((N_SKILLS, N_OCCS))

    # Score validation fold
    nb_val   = np.vstack([log_cf[sl, :].sum(0)   for sl in va_sl])
    ppmi_val = np.vstack([ppmi_f[sl, :].sum(0)    for sl in va_sl])

    ens_val = rrf_ensemble([ppmi_val, nb_val], [best_weights[0], best_weights[1]])
    fold_m  = mapk(va_ol, top5_from_scores(ens_val))
    cv_map5.append(fold_m)
    print(f'  Fold {fold+1}: mAP@5 = {fold_m:.4f}')

print(f'\nCV mAP@5: {np.mean(cv_map5):.4f} ± {np.std(cv_map5):.4f}')

## 13. Generate Test Predictions

In [ ]:
# ── Apply best weights to test score matrices ─────────────────────────
final_test_scores = rrf_ensemble(test_mats, best_weights)

# ── Decode top-5 indices → occupation codes ───────────────────────────
def decode_top5(score_row):
    top5_idx = np.argsort(score_row)[::-1][:5]
    return [idx2occ[i] for i in top5_idx]

test_predictions = [decode_top5(final_test_scores[i]) for i in range(len(test))]

# ── Build submission DataFrame ────────────────────────────────────────
sub_df = pd.DataFrame(test_predictions, columns=OCC_COLS)
sub_df.insert(0, 'ID', test['ID'].values)
sub_df.to_csv('submission_advanced.csv', index=False)

print(f'Submission saved → submission_advanced.csv')
print(f'Shape: {sub_df.shape}')
sub_df.head(10)

## 14. Submission Validation

In [ ]:
sub_check  = pd.read_csv('submission_advanced.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}SampleSubmission.csv')

assert list(sub_check.columns) == list(sample_sub.columns), '❌ Column mismatch'
assert len(sub_check) == len(test),                         '❌ Row count mismatch'
assert sub_check.isnull().sum().sum() == 0,                 '❌ Nulls found'

valid_codes = set(all_occs)
for col in OCC_COLS:
    bad = ~sub_check[col].isin(valid_codes)
    assert not bad.any(), f'❌ Invalid occ codes in {col}'

# Check no duplicate predictions per row
dupes = sub_check[OCC_COLS].apply(lambda r: len(r) != len(set(r)), axis=1).sum()
print(f'Rows with duplicate predictions: {dupes}  (should be 0)')

print('\n✅ All submission checks passed!')
print(f'   Rows: {len(sub_check):,}  |  Columns: {list(sub_check.columns)}')

## 15. Results Summary & Ablation

In [ ]:
# ── Collect all mAP@5 scores for comparison ───────────────────────────
individual_scores = {}
for name, mat in zip(scorer_names, train_mats):
    individual_scores[name] = mapk(train_oidx, top5_from_scores(mat))

print('═' * 60)
print('         SKILLS2JOB ADVANCED — RESULTS SUMMARY')
print('═' * 60)
print('  Individual Scorers (train-set mAP@5):')
for name, score in sorted(individual_scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 40)
    print(f'    {name:<18}: {score:.4f}  {bar}')
print()
print(f'  ENSEMBLE (RRF, train-set) : {best_map5:.4f}')
print(f'  ENSEMBLE (CV, 5-fold)     : {np.mean(cv_map5):.4f} ± {np.std(cv_map5):.4f}')
print('═' * 60)
print()

# ── Ablation: contribution of each scorer ────────────────────────────
print('  Ablation — remove one scorer at a time:')
for drop_idx, name in enumerate(scorer_names):
    ablated_mats = [m for i, m in enumerate(train_mats) if i != drop_idx]
    ablated_w    = [w for i, w in enumerate(best_weights) if i != drop_idx]
    if sum(ablated_w) == 0:
        continue
    ablated_w    = [w / sum(ablated_w) for w in ablated_w]  # renormalize
    ens_ab       = rrf_ensemble(ablated_mats, ablated_w)
    m_ab         = mapk(train_oidx, top5_from_scores(ens_ab))
    delta        = best_map5 - m_ab
    print(f'    Without {name:<18}: {m_ab:.4f}  (Δ = -{delta:.4f})')

print()
print('📁 Submission: submission_advanced.csv')

In [ ]:
# ── Final comparison bar chart ────────────────────────────────────────
all_methods = dict(individual_scores)
all_methods['RRF Ensemble'] = best_map5
all_methods['CV Estimate']  = np.mean(cv_map5)

sorted_m = dict(sorted(all_methods.items(), key=lambda x: x[1]))
colors   = ['#4a90d9' if k not in ('RRF Ensemble','CV Estimate') else '#e74c3c'
            for k in sorted_m]

plt.figure(figsize=(10, 5))
bars = plt.barh(list(sorted_m.keys()), list(sorted_m.values()), color=colors)
plt.axvline(0.6, color='green', linestyle='--', linewidth=1.5, label='Target: 0.60')
plt.axvline(0.3, color='orange', linestyle='--', linewidth=1.5, label='Baseline: 0.30')
for bar, v in zip(bars, sorted_m.values()):
    plt.text(v + 0.002, bar.get_y() + bar.get_height()/2,
             f'{v:.4f}', va='center', fontsize=9)
plt.xlabel('mAP@5')
plt.title('Skills2Job — Method Comparison')
plt.legend()
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

## 16. Further Improvement Tips

If mAP@5 is still below target after running this notebook, try these:

### Immediate gains
1. **Increase `N_FOLDS=10`** — more stable OOF predictions
2. **Increase `TOP_OCC`** to cover all occupations for LGB
3. **Tune `NB_ALPHA`** — lower values (0.01–0.05) usually better for dense data
4. **Tune `PMI_ALPHA`** — try 0.5–1.0 range

### Advanced techniques
5. **Matrix Factorization** — SVD/NMF on the skill→occ matrix gives latent embeddings
6. **Graph Neural Network** — model skill–occupation as a bipartite graph
7. **Word2Vec** on skill sequences — treat each training row as a 'sentence' of skill codes
8. **Isotonic regression calibration** — calibrate LGB probabilities post-training
9. **Bayesian weight optimization** — use Optuna instead of grid search

### Data augmentation
10. **Skill permutation** — each row with 5 skills has 5! = 120 orderings; use all
11. **Sub-sampling** — generate pseudo-samples with 4 skills from each 5-skill row